# PyTorch 自动微分（AutoGrad）

## 1. 环境初始化与依赖导入

In [ ]:
import torch  # 导入 PyTorch 深度学习框架，本章主要使用其自动微分（autograd）功能

## 2. 背景：PyTorch 的自动微分机制

参考文档：[PyTorch Autograd Tutorial](https://pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html)

PyTorch 实现模型训练需要完整地写下训练过程，包括：
1. **前向传播**：计算预测值和损失
2. **反向传播**：通过 `loss.backward()` 自动计算梯度
3. **参数更新**：通过优化器（如 SGD）应用梯度下降

本章先从**数值近似求导**开始，再介绍 PyTorch 的**符号自动微分**（`torch.autograd`）。

## 3. 数值近似求导

### 3.1 函数定义与一元函数的近似导数

In [ ]:
# 定义测试函数 f(x) = 3x² + 2x - 1
def f(x):
    """一元多项式函数，解析导数为 f'(x) = 6x + 2
    参数:
        x (float | Tensor): 自变量
    返回值:
        与 x 同类型的函数值
    """
    return 3. * x ** 2 + 2. * x - 1  # 计算多项式值

# 数值近似求导：用差商 [f(x+ε) - f(x-ε)] / (2ε) 近似 f'(x)（中心差分法）
# 中心差分比前向差分精度更高（误差为 O(ε²) 而非 O(ε)）
def approximate_derivative(f, x, eps=1e-6):
    """用中心差分法数值近似计算函数 f 在点 x 处的导数
    参数:
        f   (callable): 待求导的函数
        x   (float)   : 求导点
        eps (float)   : 差分步长，默认 1e-6；越小精度越高，但过小会有浮点误差
    返回值:
        float：f 在 x 处的近似导数值
    """
    return (f(x + eps) - f(x - eps)) / (2. * eps)  # 中心差分公式

# 验证：f'(1) = 6×1 + 2 = 8，近似结果应接近 8
print(approximate_derivative(f, 1.))  # 输出：约 7.999999...

7.999999999785956


### 3.2 多元函数的偏导数数值近似

In [ ]:
# 定义二元函数 g(x1, x2) = (x1 + 5) * x2²
# 解析偏导数：∂g/∂x1 = x2²，∂g/∂x2 = 2 * (x1 + 5) * x2
# 在 (x1=2, x2=3) 处：∂g/∂x1 = 9，∂g/∂x2 = 42
def g(x1, x2):
    """二元函数
    参数:
        x1 (float): 第一个自变量
        x2 (float): 第二个自变量
    返回值:
        float: 函数值
    """
    return (x1 + 5) * (x2 ** 2)  # 计算二元函数值

# 数值近似计算偏导数：固定一个变量，对另一个变量求近似导数
def approximate_gradient(g, x1, x2, eps=1e-3):
    """数值近似计算 g(x1, x2) 对 x1 和 x2 的偏导数
    参数:
        g   (callable): 二元函数
        x1  (float)   : 第一个自变量
        x2  (float)   : 第二个自变量
        eps (float)   : 差分步长；默认 1e-3
    返回值:
        tuple: (∂g/∂x1, ∂g/∂x2)，各为 float
    """
    # 固定 x2，对 x1 用 lambda 封装为一元函数后求近似导数
    dg_x1 = approximate_derivative(lambda x: g(x, x2), x1, eps)  # ∂g/∂x1 近似值
    # 固定 x1，对 x2 求近似导数
    dg_x2 = approximate_derivative(lambda x: g(x1, x), x2, eps)  # ∂g/∂x2 近似值
    return dg_x1, dg_x2  # 返回两个偏导数的近似值

# 验证：在 (2, 3) 处，∂g/∂x1 应约为 9，∂g/∂x2 应约为 42
print(approximate_gradient(g, 2., 3.))

(8.999999999993236, 41.999999999994486)


## 4. PyTorch 自动微分（torch.autograd）

### 4.1 使用 autograd.grad 计算单变量偏导数

In [ ]:
# 使用 torch.autograd.grad 精确计算偏导数（符号微分，而非数值近似）
# requires_grad=True：告知 PyTorch 需要对该 Tensor 进行求导追踪
x1 = torch.tensor([2.], requires_grad=True)  # 创建需要求导的 Tensor，值为 [2.]，shape=[1]
x2 = torch.tensor([3.], requires_grad=True)  # 创建需要求导的 Tensor，值为 [3.]，shape=[1]
y = g(x1, x2)  # 计算 y = g(x1, x2)；PyTorch 自动构建计算图（computational graph）

# torch.autograd.grad(outputs, inputs, retain_graph):
#   outputs       : 要对其求导的输出 Tensor（标量）
#   inputs        : 要求偏导的输入 Tensor（或列表）
#   retain_graph  : 是否保留计算图（默认释放）；若后续还需对同一图求导，需设为 True
#   返回值        : tuple of Tensor，与 inputs 等长
(dy_dx1,) = torch.autograd.grad(y, x1, retain_graph=True)  # 计算 ∂y/∂x1；结果应为 tensor([9.])
print(dy_dx1)  # 输出：tensor([9.])，与解析结果 x2²=9 一致

tensor([9.])


### 4.2 retain_graph 参数的作用

In [ ]:
# 如果不设置 retain_graph=True，第一次调用 grad() 后计算图会被释放
# 第二次调用时会因计算图不存在而报错
try:  # 使用 try-except 捕获可能的异常
    # retain_graph=True：保留计算图，允许对同一 y 多次求导
    (dy_dx2,) = torch.autograd.grad(y, x2, retain_graph=True)  # 计算 ∂y/∂x2
    print(dy_dx2)  # 输出：tensor([42.])，与解析结果 2*(x1+5)*x2 = 2*7*3 = 42 一致
except Exception as e:
    print(e)  # 若不加 retain_graph=True，此处会打印计算图已释放的错误信息

tensor([42.])


### 4.3 同时计算多变量偏导数

In [ ]:
# 更简洁的写法：在 inputs 参数中同时传入多个变量，一次性计算所有偏导数
x1 = torch.tensor([2.], requires_grad=True)  # 重新创建（避免使用上面已计算的版本）
x2 = torch.tensor([3.], requires_grad=True)
y  = g(x1, x2)  # 重新构建计算图

# inputs 参数为列表：[x1, x2]，同时计算两个偏导数
# 返回值 dy_dx1, dy_dx2 均为 Tensor
dy_dx1, dy_dx2 = torch.autograd.grad(y, [x1, x2])  # 一次调用同时得到两个偏导数

print(dy_dx1, dy_dx2)  # 输出：tensor([9.]) tensor([42.])

tensor([9.]) tensor([42.])


### 4.4 使用 backward() 方法计算梯度（常用方式）

In [ ]:
# backward() 是训练神经网络时最常用的求梯度方式
# 调用后，梯度会累积到各 Tensor 的 .grad 属性中
x1 = torch.tensor([2.], requires_grad=True)  # 需要求导的 Tensor
x2 = torch.tensor([3.], requires_grad=True)  # 需要求导的 Tensor
y  = g(x1, x2)  # 前向计算，构建计算图

# y.backward()：对 y 关于所有 requires_grad=True 的 Tensor 执行反向传播
# 计算结果存入各 Tensor 的 .grad 属性（累积方式，多次调用会叠加，需手动清零）
y.backward()  # 执行反向传播，计算所有叶子节点的梯度

# 梯度存储在 .grad 属性中；梯度下降更新公式：x -= learning_rate * x.grad
print(x1.grad, x2.grad)  # 输出：tensor([9.]) tensor([42.])

tensor([9.]) tensor([42.])


## 5. 二阶导数

In [ ]:
# 计算二阶偏导数（Hessian 矩阵元素），需要 create_graph=True 保留一阶导数的计算图
x1 = torch.tensor([2.], requires_grad=True)  # 需要求导的 Tensor
x2 = torch.tensor([3.], requires_grad=True)  # 需要求导的 Tensor
y  = g(x1, x2)  # 前向计算

# 第一步：计算一阶偏导数，并保留一阶导数的计算图（create_graph=True）
# create_graph=True：构建高阶导数所需的计算图（否则 dy_dx1 本身不可微）
dy_dx1, dy_dx2 = torch.autograd.grad(y, [x1, x2], create_graph=True)

# 第二步：对一阶偏导数再次求导，得到二阶偏导数
# allow_unused=True：允许 inputs 中某些 Tensor 与 outputs 无关，此时返回 None
dy_dx1_dx1, dy_dx1_dx2 = torch.autograd.grad(dy_dx1, [x1, x2], allow_unused=True)  # ∂²y/∂x1², ∂²y/∂x1∂x2
dy_dx2_dx1, dy_dx2_dx2 = torch.autograd.grad(dy_dx2, [x1, x2], allow_unused=True)  # ∂²y/∂x2∂x1, ∂²y/∂x2²

# g(x1,x2)=(x1+5)*x2² → ∂²g/∂x1²=0（None），∂²g/∂x2∂x1=2*x2=6，∂²g/∂x2²=2*(x1+5)=14
print(dy_dx1_dx1, dy_dx2_dx1, dy_dx2_dx2)  # 输出：None tensor([6.]) tensor([14.])

None tensor([6.]) tensor([14.])


## 6. 模拟梯度下降算法（SGD）

### 6.1 手动实现 SGD（不使用优化器）

In [ ]:
# 手动实现 SGD 梯度下降，求 f(x) = 3x² + 2x - 1 的极小值点
# 解析极小值：f'(x) = 6x + 2 = 0 → x = -1/3 ≈ -0.3333
import torch  # 重新导入确保 torch 可用
learning_rate = 0.3  # 学习率（步长），控制每步参数更新的幅度
x = torch.tensor(2.0, requires_grad=True)  # 初始化参数 x=2.0，requires_grad=True 开启梯度追踪

for _ in range(100):   # 迭代 100 次
    z = f(x)            # 前向计算：计算损失（函数值）
    if _ > 0:            # 从第 2 次迭代起清零梯度（第 1 次无梯度可清）
        x.grad.zero_()  # 原地将梯度清零；等价于 x.grad -= x.grad；防止梯度累积
    z.backward()         # 反向传播：计算 dz/dx，结果存入 x.grad
    # 参数更新：x -= learning_rate * x.grad（梯度下降公式）
    # .data.sub_() 直接操作底层数据，不影响计算图和 requires_grad 属性
    x.data.sub_(learning_rate * x.grad)  # 等价于 optimizer.step()

print(x)  # 收敛后输出 x ≈ -0.3333

tensor(-0.3333, requires_grad=True)


In [ ]:
a = torch.tensor(2)  # 创建标量整数 Tensor（不含 requires_grad）
a.shape  # .shape 属性：返回形状，标量 Tensor 的 shape 为 torch.Size([])（空 Size）

torch.Size([])

### 6.2 结合 PyTorch 优化器（torch.optim.SGD）

In [ ]:
# 使用 torch.optim.SGD 优化器简化梯度下降流程（与上面手动实现等价）
learning_rate = 0.3  # 学习率
x = torch.tensor(2.0, requires_grad=True)  # 初始化参数 x=2.0

# 创建 SGD 优化器
# 参数:
#   params    : 需要优化的参数列表；这里传入 [x]
#   lr        : 学习率（learning rate）
#   momentum  : 动量系数（0.9 表示保留上一步梯度的 90%），加速收敛并减少震荡
optimizer = torch.optim.SGD([x], lr=learning_rate, momentum=0.9)

for _ in range(220):       # 使用动量需要更多步才能收敛（因为动量导致超调）
    z = f(x)                # 前向计算
    optimizer.zero_grad()   # 清零梯度（等价于手动 x.grad.zero_()）
    z.backward()             # 反向传播：计算 dz/dx，结果存入 x.grad
    optimizer.step()         # 参数更新：x -= lr * x.grad（含动量调整）

print(x)  # 收敛后输出 x ≈ -0.3333，与手动实现结果相同

tensor(-0.3333, requires_grad=True)
